In [0]:
# List the built-in NYC Taxi files
display(dbutils.fs.ls("/databricks-datasets/nyctaxi/tripdata/yellow/"))

In [0]:
from pyspark.sql.functions import col, to_timestamp, hour, dayofweek

# ==========================================
# 1. BRONZE LAYER (Raw Ingestion)
# ==========================================
# We read just two months of data so the free cluster computes it quickly
raw_taxi_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2019-0[1-2].csv.gz")

# Save to your permanent local managed Delta Lake
raw_taxi_df.write.format("delta").mode("overwrite").saveAsTable("bronze_nyc_taxi")

# ==========================================
# 2. SILVER LAYER (Data Cleaning & Feature Engineering)
# ==========================================
# Read from bronze, clean up bad data, and create time-based features
silver_df = spark.table("bronze_nyc_taxi") \
    .filter(col("passenger_count") > 0) \
    .filter(col("trip_distance") > 0.0) \
    .filter(col("fare_amount") > 0.0) \
    .dropna(subset=["tpep_pickup_datetime", "tpep_dropoff_datetime"]) \
    .withColumn("pickup_time", to_timestamp(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_hour", hour(col("pickup_time"))) \
    .withColumn("pickup_day", dayofweek(col("pickup_time")))

silver_df.write.format("delta").mode("overwrite").saveAsTable("silver_nyc_taxi")

# ==========================================
# 3. GOLD LAYER (Business Intelligence Aggregation)
# ==========================================
# Run an aggregation to find the average fare and trip distance by hour of day
gold_df = spark.sql("""
    SELECT pickup_hour, 
           COUNT(*) as total_trips, 
           ROUND(AVG(fare_amount), 2) as avg_fare, 
           ROUND(AVG(trip_distance), 2) as avg_distance
    FROM silver_nyc_taxi
    GROUP BY pickup_hour
    ORDER BY pickup_hour
""")

gold_df.write.format("delta").mode("overwrite").saveAsTable("gold_nyc_taxi_metrics")

# Look at your final business-ready dataset!
display(spark.table("gold_nyc_taxi_metrics"))

In [0]:
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
# 1. FIXED IMPORT: Import the dedicated root_mean_squared_error function
from sklearn.metrics import root_mean_squared_error

# Prepare data (sampling a subset so it easily fits into the free tier's memory)
ml_data = spark.table("silver_nyc_taxi").select("pickup_hour", "trip_distance", "fare_amount").limit(50000).toPandas()
X = ml_data[["pickup_hour", "trip_distance"]]
y = ml_data["fare_amount"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Track the experiment
with mlflow.start_run():
    # Train a simple baseline regression model
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    # Evaluate
    predictions = model.predict(X_test)
    
    # 2. FIXED METRIC: Call the clean function without any extra arguments
    rmse = root_mean_squared_error(y_test, predictions)
    
    # Log parameters, metrics, and the model to the UI
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_metric("rmse", rmse)
    mlflow.sklearn.log_model(model, "taxi_reg_model")
    
    print(f"Model trained successfully! Testing RMSE: {rmse}")
